# 05 - Merge de SISAM al dataset combinado (DETER + BDQueimadas)
### Proyecto BIODIVERSITY-GUARD-Predict (1ACC0057 - Machine Learning)

## CORRECCION IMPORTANTE (v2): la clave de merge es `(municipio_norm, uf)`, no solo `municipio_norm`

Se detecto en la primera version de este notebook que usar solo `municipio_norm` como clave duplicaba filas: Brasil tiene municipios con el mismo nombre en distintos estados (ej. `PAU D'ARCO` existe en Para y en Tocantins), y SISAM cubre 100% de los municipios de la Amazonia Legal, por lo que la ambiguedad se manifiesta con mas fuerza que en BDQueimadas. Esta version corrige la clave en todos los `groupby` y `merge`.

**IMPORTANTE:** esta misma correccion tuvo que aplicarse retroactivamente al Notebook 03 (se detecto que 1 municipio de los 424 de DETER, `PAU D'ARCO`, tenia el mismo problema). Este notebook asume que ya estas usando la version corregida de `dataset_alertas_frente2.csv` y `dataset_semanal_frente1.csv` (regeneradas con la clave `(municipio_norm, uf)`), no la version anterior.

## Que hace este notebook

Se parte de los datasets ya validados y corregidos del Notebook 3 y se les agrega SISAM como tercera fuente, con la misma logica de granularidad:

| | Frente 2 (`dataset_alertas`) | Frente 1 (`dataset_semanal`) |
|---|---|---|
| Ventana de SISAM | 7 dias previos a la fecha exacta de cada alerta | Promedio de la semana calendario completa |

SISAM no necesita grilla de fechas (a diferencia de BDQueimadas): es reanalisis modelado, con un valor todos los dias para todos los municipios.

## Archivos de entrada requeridos
1. `dataset_alertas_frente2.csv` - version corregida del Notebook 3 (incluye columna `uf`)
2. `dataset_semanal_frente1.csv` - version corregida del Notebook 3 (incluye columna `uf`)
3. `sisam_limpio.csv` - salida del Notebook 4

In [ ]:
import pandas as pd
import numpy as np

## Paso 1: Cargar los 3 archivos y unificar el codigo de estado (`uf`) en SISAM

In [ ]:
dataset_alertas = pd.read_csv("dataset_alertas_frente2.csv")
dataset_alertas["view_date"] = pd.to_datetime(dataset_alertas["view_date"])

dataset_semanal = pd.read_csv("dataset_semanal_frente1.csv")
dataset_semanal["semana"] = pd.to_datetime(dataset_semanal["semana"])

df_sisam = pd.read_csv("sisam_limpio.csv")
df_sisam["data"] = pd.to_datetime(df_sisam["data"])

# SISAM trae "estado" como nombre completo -> mapear a la abreviatura ("uf")
mapa_uf = {
    "ACRE": "AC", "AMAPÁ": "AP", "AMAZONAS": "AM", "MARANHÃO": "MA",
    "MATO GROSSO": "MT", "PARÁ": "PA", "RONDÔNIA": "RO",
    "RORAIMA": "RR", "TOCANTINS": "TO"
}
df_sisam["uf"] = df_sisam["estado"].map(mapa_uf)

print("Estados de SISAM sin mapear (debe ser 0):", df_sisam["uf"].isna().sum())

assert "uf" in dataset_alertas.columns, "dataset_alertas no tiene columna 'uf' - usa la version corregida del Notebook 3"
assert "uf" in dataset_semanal.columns, "dataset_semanal no tiene columna 'uf' - usa la version corregida del Notebook 3"

print("\ndataset_alertas (antes de SISAM):", dataset_alertas.shape)
print("dataset_semanal (antes de SISAM):", dataset_semanal.shape)
print("SISAM (nivel registro diario):", df_sisam.shape)

print("\nNivel_Riesgo_Amenaza (antes):")
print(dataset_alertas["Nivel_Riesgo_Amenaza"].value_counts())
print("\narea_ha_total describe (antes):")
print(dataset_semanal["area_ha_total"].describe())

Estados de SISAM sin mapear (debe ser 0): 0

dataset_alertas (antes de SISAM): (49971, 24)
dataset_semanal (antes de SISAM): (11445, 12)
SISAM (nivel registro diario): (885934, 10)

Nivel_Riesgo_Amenaza (antes):
Nivel_Riesgo_Amenaza
Moderado    32440
Alto         7743
Bajo         6483
Crítico      3305
Name: count, dtype: int64

area_ha_total describe (antes):
count    11445.000000
mean       291.371357
std       1219.044315
min          0.010000
25%         16.140000
50%         45.020000
75%        163.380000
max      55192.060000
Name: area_ha_total, dtype: float64


## Paso 2: Enriquecer `dataset_alertas` (Frente 2) con SISAM

### 2.1 Ventana movil de 7 dias hacia atras, por (municipio, estado)

In [ ]:
cols_sisam = ["pm10_reanalise", "pm2_5_reanalise", "o3_reanalise",
              "no2_reanalise", "co_reanalise", "so2_reanalise"]

sisam_ordenado = df_sisam.sort_values(["municipio_norm", "uf", "data"]).set_index("data")

sisam_rolling = (
    sisam_ordenado.groupby(["municipio_norm", "uf"])[cols_sisam]
    .rolling("7D", min_periods=1).mean()
    .reset_index()
)

sisam_rolling = sisam_rolling.rename(columns={c: f"{c}_7d_previo" for c in cols_sisam})

print(sisam_rolling.shape)
sisam_rolling.head()

(885934, 9)


,municipio_norm,uf,data,pm10_reanalise_7d_previo,pm2_5_reanalise_7d_previo,o3_reanalise_7d_previo,no2_reanalise_7d_previo,co_reanalise_7d_previo,so2_reanalise_7d_previo
0,ABAETETUBA,PA,2022-01-01,17.4100,13.140,38.620000,0.4900,0.130000,0.330000
1,ABAETETUBA,PA,2022-01-02,17.3650,13.270,39.085000,0.4750,0.125000,0.370000
2,ABAETETUBA,PA,2022-01-03,15.3700,11.640,37.303333,0.4900,0.123333,0.386667
3,ABAETETUBA,PA,2022-01-04,14.6775,11.020,36.772500,0.4575,0.122500,0.355000
4,ABAETETUBA,PA,2022-01-05,15.0440,11.258,35.748000,0.4780,0.122000,0.374000


### 2.2 Unir a cada alerta individual (por fecha exacta + municipio + estado)

In [ ]:
dataset_alertas = dataset_alertas.merge(
    sisam_rolling,
    left_on=["view_date", "municipio_norm", "uf"],
    right_on=["data", "municipio_norm", "uf"],
    how="left"
).drop(columns=["data"])

print("dataset_alertas (con SISAM):", dataset_alertas.shape)
assert dataset_alertas.shape[0] == 49971, f"ALERTA: cambio el numero de filas: {dataset_alertas.shape[0]}"

print("\nNivel_Riesgo_Amenaza se conserva intacto:")
print(dataset_alertas["Nivel_Riesgo_Amenaza"].value_counts())

cols_sisam_7d = [c for c in dataset_alertas.columns if c.endswith("_reanalise_7d_previo")]
print("\nNulos en las nuevas columnas de SISAM:")
print(dataset_alertas[cols_sisam_7d].isnull().sum())

dataset_alertas (con SISAM): (49971, 30)

Nivel_Riesgo_Amenaza se conserva intacto:
Nivel_Riesgo_Amenaza
Moderado    32440
Alto         7743
Bajo         6483
Crítico      3305
Name: count, dtype: int64

Nulos en las nuevas columnas de SISAM:
pm10_reanalise_7d_previo     0
pm2_5_reanalise_7d_previo    0
o3_reanalise_7d_previo       0
no2_reanalise_7d_previo      0
co_reanalise_7d_previo       0
so2_reanalise_7d_previo      0
dtype: int64


## Paso 3: Enriquecer `dataset_semanal` (Frente 1) con SISAM

In [ ]:
df_sisam["semana"] = df_sisam["data"].dt.to_period("W").dt.start_time

sisam_semanal = (
    df_sisam
    .groupby(["semana", "municipio_norm", "uf"])[cols_sisam]
    .mean()
    .reset_index()
)

print(sisam_semanal.shape)
sisam_semanal.head()

(127717, 9)


,semana,municipio_norm,uf,pm10_reanalise,pm2_5_reanalise,o3_reanalise,no2_reanalise,co_reanalise,so2_reanalise
0,2021-12-27,ABAETETUBA,PA,17.365,13.270,39.085,0.475,0.125,0.37
1,2021-12-27,ABEL FIGUEIREDO,PA,19.445,14.300,23.795,1.625,0.170,0.25
2,2021-12-27,ABREULANDIA,TO,16.185,12.110,21.960,0.640,0.130,0.08
3,2021-12-27,ACAILANDIA,MA,14.000,10.230,23.040,1.190,0.135,0.19
4,2021-12-27,ACARA,PA,16.920,12.855,37.030,0.560,0.125,0.36


In [ ]:
dataset_semanal = dataset_semanal.merge(
    sisam_semanal,
    on=["semana", "municipio_norm", "uf"],
    how="left"
)

print("dataset_semanal (con SISAM):", dataset_semanal.shape)
assert dataset_semanal.shape[0] == 11445, f"ALERTA: cambio el numero de filas: {dataset_semanal.shape[0]}"

print("\narea_ha_total describe (debe ser identico al de antes):")
print(dataset_semanal["area_ha_total"].describe())

print("\nNulos en las nuevas columnas de SISAM:")
print(dataset_semanal[cols_sisam].isnull().sum())

dataset_semanal (con SISAM): (11445, 18)

area_ha_total describe (debe ser identico al de antes):
count    11445.000000
mean       291.371357
std       1219.044315
min          0.010000
25%         16.140000
50%         45.020000
75%        163.380000
max      55192.060000
Name: area_ha_total, dtype: float64

Nulos en las nuevas columnas de SISAM:
pm10_reanalise     0
pm2_5_reanalise    0
o3_reanalise       0
no2_reanalise      0
co_reanalise       0
so2_reanalise      0
dtype: int64


**Nota para el Modelado:** estas variables de SISAM tambien deben *lagearse* en Feature Engineering si el objetivo es forecasting a horizonte 7/30 dias, igual que las de BDQueimadas.

## Paso 4: Guardar los datasets finales (DETER + BDQueimadas + SISAM)

In [ ]:
dataset_alertas.to_csv("dataset_alertas_final.csv", index=False, encoding="utf-8")
dataset_semanal.to_csv("dataset_semanal_final.csv", index=False, encoding="utf-8")

print("Guardado: dataset_alertas_final.csv  ->", dataset_alertas.shape)
print("Guardado: dataset_semanal_final.csv  ->", dataset_semanal.shape)

Guardado: dataset_alertas_final.csv  -> (49971, 30)
Guardado: dataset_semanal_final.csv  -> (11445, 18)
